# WM-Audit: 10-minute audit

Run the full battery on the built-in light env: train a tiny world model, compute pixel / control / uncertainty diagnostics, close-loop control, and correlate diagnostics with closed-loop return.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from wm_audit.audit import evaluate_config
from wm_audit.diagnostics import audit_correlations, prereg_decision

In [ ]:
cfg = dict(buffer_capacity=2000, collect_steps=1500, data_seed=1,
           ensemble_size=2, train_steps=80, batch_size=32, lr=2e-3,
           calib_segments=12, calib_horizon=6, cf_pairs=20,
           cem_horizon=4, cem_candidates=24, cem_elites=6, cem_iters=2)
rows = []
for seed in [0, 1]:
    for hidden, embed in [(16, 16), (32, 16)]:
        rows.append(evaluate_config('pixel_nav', hidden, embed, seed, cfg))
rows

In [ ]:
diag = ['pixel_mse','pixel_ssim','return_ece','counterfactual_tau','uncertainty_rho']
res = audit_correlations(rows, diag, return_key='R_real', n_boot=200)
for d, v in res.items():
    print(f"{d:20s} rho={v['rho']:+.3f} CI=[{v['ci_lo']:+.3f},{v['ci_hi']:+.3f}] p_adj={v['p_adj']:.3f}")
print(prereg_decision(res['pixel_mse']['rho']))